## Objectifs du notebook

- **Chargement et jointure des tables** : fusion des CSV en un seul dataset
- **Cibles** : deux variables à prédire (top 10 et position d'arrivée)
- **Retrait des fuites** : éviter que le modèle triche en utilisant les données futures
- **Feature engineering** : transformer les données brutes en features exploitables
- **Contrôles et sauvegarde** : doublons, valeurs manquantes, export du dataset final

<br/>

## Résultat attendu : f1_table.csv

- Organisation : une ligne par pilote et par course (+2006)
- Informations : uniquement des infos connues avant le départ
- Données parasites : aucun doublon, aucune colonne post-course


In [ ]:
# ÉTAPE 1 : CHARGEMENT ET JOINTURES

# Import
import pandas as pd


# On ne garde que les colonnes qui nous intéressent (et valeur manquante = \N)
results = pd.read_csv("../data/raw-data/results.csv", na_values="\\N")
races = pd.read_csv("../data/raw-data/races.csv", na_values="\\N")[["raceId", "year", "round", "circuitId", "date"]]
status = pd.read_csv("../data/raw-data/status.csv")
drivers = pd.read_csv("../data/raw-data/drivers.csv", na_values="\\N")[["driverId", "driverRef", "dob"]]
constructors = pd.read_csv("../data/raw-data/constructors.csv")[["constructorId", "constructorRef"]]
circuits = pd.read_csv("../data/raw-data/circuits.csv")[["circuitId", "circuitRef"]]
quali = pd.read_csv("../data/raw-data/qualifying.csv", na_values="\\N")[["raceId", "driverId", "position", "q1", "q2", "q3"]]
quali = quali.rename(columns={"position": "quali_pos"})


# Jointure des 7 tables en un unique dataset 
df = results.merge(races, on="raceId")
df = df.merge(status, on="statusId")
df = df.merge(drivers, on="driverId")
df = df.merge(constructors, on="constructorId")
df = df.merge(circuits, on="circuitId")
df = df.merge(quali, on=["raceId", "driverId"], how="left")   # left : pas de qualif avant 1994


# Conversion des colonnes texte en date 
df["date"] = pd.to_datetime(df["date"])
df["dob"] = pd.to_datetime(df["dob"])


# Tests (nombre et affichages)
print(len(results), len(df))   # les deux nombres doivent être égaux
df.head()